# Mother prediction interface guide

This notebook provides an overview of how Mother exposes `predict()` and `predict_uncertainty()` across estimators.
It focuses on practical usage rather than implementation details.
All examples use small, freely available datasets from scikit-learn, so the notebook is self-contained and easy to adapt.

### Core concepts before you start

A few rules make the interface easier to use in practice:
- `predict()` returns the model's point predictions in the estimator's native shape.
- `predict_uncertainty()` attempts to return a standardized pandas DataFrame, even when the underlying model exposes uncertainty differently.
- Some estimators implement custom uncertainty logic (e.g. CatboostRegressorMother, CatboostClassifierMother, RandomForestRegressorMother). Others inherit the generic fallback from `AbstractMotherPipeline`, which wraps `predict()` output into a standard schema.
- In single-target mode, outputs usually expose columns such as `pred`, `mean_predictions`, `knowledge_uncertainty`, `data_uncertainty`, and `total_uncertainty`.
- In multi-target mode, the same concepts are namespaced per target, for example `target_0_pred` and `target_1_total_uncertainty`.
- Fallback output is best interpreted as interface normalization. It should not automatically be read as evidence that the estimator has a rich uncertainty model.

The next sections show these patterns side by side.

In [1]:
import warnings

import pandas as pd
from IPython.display import display
from sklearn.datasets import load_breast_cancer, load_diabetes, make_multilabel_classification, make_regression
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import KFold, train_test_split
from sklearn.multioutput import MultiOutputClassifier

from mother import ml
from mother.ml.models.m_catboost import (
    CatboostClassifierMother,
    CatboostGaussianProcessRegressorMother,
    CatboostRegressorMother,
)
from mother.ml.models.m_lasso import LassoClassifierMother, LassoRegressorMother
from mother.ml.models.m_randomForest import RandomForestClassifierMother, RandomForestRegressorMother
from mother.pipeline_utils import mother_cv

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 140)

In [2]:
def interface_summary(name, pred_output, uncertainty_output):
    pred_shape = getattr(pred_output, "shape", (len(pred_output),))
    summary = {
        "estimator": name,
        "predict_type": type(pred_output).__name__,
        "predict_shape": tuple(pred_shape),
        "uncertainty_type": type(uncertainty_output).__name__,
        "uncertainty_shape": tuple(uncertainty_output.shape),
        "uncertainty_columns": list(uncertainty_output.columns),
    }
    return summary


def show_output(name, pred_output, uncertainty_output, head_rows=3):
    print(f"\n=== {name} ===")
    print("predict() type:", type(pred_output).__name__)
    print("predict() shape:", getattr(pred_output, "shape", (len(pred_output),)))
    print("predict_uncertainty() shape:", uncertainty_output.shape)
    print("predict_uncertainty() columns:")
    print(list(uncertainty_output.columns))
    display(uncertainty_output.head(head_rows))

## 1. Single-target regression

We compare four regressors:

- `LassoRegressorMother`: typically falls back to the generic uncertainty interface
- `RandomForestRegressorMother`: custom uncertainty implementation
- `CatboostRegressorMother`: custom uncertainty implementation
- `CatboostGaussianProcessRegressorMother`: custom uncertainty implementation based on CatBoost Gaussian process sampling

In [3]:
X_reg, y_reg = load_diabetes(return_X_y=True, as_frame=True)
X_reg_train, X_reg_test, y_reg_train, y_reg_test = train_test_split(X_reg, y_reg, test_size=0.2, random_state=42)

single_target_regressors = {
    "Lasso_fallback": LassoRegressorMother(),
    "RandomForest_custom": RandomForestRegressorMother(n_estimators=40, random_state=42),
    "CatBoost_custom": CatboostRegressorMother(target_type="single_target", iterations=40, verbose=0),
    "CatBoostGaussianProcess_custom": CatboostGaussianProcessRegressorMother(
        samples=5, prior_iterations=50, learning_rate=0.1, verbose=False
    ),
}

single_target_regression_summaries = []
for name, model in single_target_regressors.items():
    model.fit(X_reg_train, y_reg_train)
    pred_output = model.predict(X_reg_test)
    uncertainty_output = model.predict_uncertainty(X_reg_test)
    single_target_regression_summaries.append(interface_summary(name, pred_output, uncertainty_output))
    show_output(name, pred_output, uncertainty_output)

pd.DataFrame(single_target_regression_summaries)[["estimator", "predict_shape", "uncertainty_shape"]]

No loss_function specified. Using default loss function based on target type.
Uncertainty quantification is not implemented for LassoRegressorMother. predict() will be used as a fallback.



=== Lasso_fallback ===
predict() type: ndarray
predict() shape: (89,)
predict_uncertainty() shape: (89, 5)
predict_uncertainty() columns:
['pred', 'mean_predictions', 'knowledge_uncertainty', 'data_uncertainty', 'total_uncertainty']


,pred,mean_predictions,knowledge_uncertainty,data_uncertainty,total_uncertainty
287,157.913466,None,None,None,None
211,162.871413,None,None,None,None
72,172.363821,None,None,None,None



=== RandomForest_custom ===
predict() type: ndarray
predict() shape: (89,)
predict_uncertainty() shape: (89, 5)
predict_uncertainty() columns:
['pred', 'mean_predictions', 'knowledge_uncertainty', 'data_uncertainty', 'total_uncertainty']


,pred,mean_predictions,knowledge_uncertainty,data_uncertainty,total_uncertainty
287,167.850,None,None,None,67.25
211,172.800,None,None,None,79.25
72,169.725,None,None,None,80.00



=== CatBoost_custom ===
predict() type: ndarray
predict() shape: (89,)
predict_uncertainty() shape: (89, 5)
predict_uncertainty() columns:
['pred', 'mean_predictions', 'knowledge_uncertainty', 'data_uncertainty', 'total_uncertainty']


,pred,mean_predictions,knowledge_uncertainty,data_uncertainty,total_uncertainty
287,151.679595,152.952556,0.687791,None,None
211,156.527527,156.702275,0.754139,None,None
72,157.828683,157.384846,0.570132,None,None


Parameter prior_iterations set to: 50, automatically updating posterior_iterations to: 950.



=== CatBoostGaussianProcess_custom ===
predict() type: ndarray
predict() shape: (89,)
predict_uncertainty() shape: (89, 5)
predict_uncertainty() columns:
['pred', 'mean_predictions', 'knowledge_uncertainty', 'data_uncertainty', 'total_uncertainty']


,pred,mean_predictions,knowledge_uncertainty,data_uncertainty,total_uncertainty
287,156.955121,None,12.211534,None,None
211,171.078571,None,14.786957,None,None
72,152.291213,None,6.435196,None,None


,estimator,predict_shape,uncertainty_shape
0,Lasso_fallback,"(89,)","(89, 5)"
1,RandomForest_custom,"(89,)","(89, 5)"
2,CatBoost_custom,"(89,)","(89, 5)"
3,CatBoostGaussianProcess_custom,"(89,)","(89, 5)"


### Observation

Even when the internal uncertainty logic differs, the public `predict_uncertainty()` interface tries to stay regular: regression outputs expose `pred`, `mean_predictions`, `knowledge_uncertainty`, `data_uncertainty`, and `total_uncertainty`.

For `CatboostGaussianProcessRegressorMother`, keep in mind that only `target_type="single_target"` is supported. Multi-target usage is intentionally rejected by the estimator.

## 2. Single-target classification

For classifiers, Mother also exposes class probabilities in many uncertainty outputs.

We compare:

- `RandomForestClassifierMother`: generic fallback with `predict_proba()`-based entropy
- `LassoClassifierMother`: generic fallback with `predict_proba()`-based entropy
- `CatboostClassifierMother`: custom uncertainty implementation

In [4]:
X_clf, y_clf = load_breast_cancer(return_X_y=True, as_frame=True)
X_clf_train, X_clf_test, y_clf_train, y_clf_test = train_test_split(X_clf, y_clf, test_size=0.2, random_state=42)

single_target_classifiers = {
    "RandomForest_fallback_plus_entropy": RandomForestClassifierMother(n_estimators=40, random_state=42),
    "Lasso_fallback_plus_entropy": LassoClassifierMother(),
    "CatBoost_custom": CatboostClassifierMother(
        target_type="single_target", model_type="classification_binary", iterations=40, learning_rate=0.1, verbose=0
    ),
}

single_target_classification_summaries = []
for name, model in single_target_classifiers.items():
    model.fit(X_clf_train, y_clf_train)
    pred_output = model.predict(X_clf_test)
    uncertainty_output = model.predict_uncertainty(X_clf_test)
    single_target_classification_summaries.append(interface_summary(name, pred_output, uncertainty_output))
    show_output(name, pred_output, uncertainty_output)

pd.DataFrame(single_target_classification_summaries)[["estimator", "predict_shape", "uncertainty_shape"]]

Uncertainty quantification is not implemented for RandomForestClassifierMother. predict() will be used as a fallback.
Uncertainty quantification is not implemented for RandomForestClassifierMother. predict() predict_proba() will be used as a fallback with entropy-based uncertainty for classification tasks.



=== RandomForest_fallback_plus_entropy ===
predict() type: ndarray
predict() shape: (114,)
predict_uncertainty() shape: (114, 7)
predict_uncertainty() columns:
['pred', 'proba_0', 'proba_1', 'mean_predictions', 'knowledge_uncertainty', 'data_uncertainty', 'total_uncertainty']


,pred,proba_0,proba_1,mean_predictions,knowledge_uncertainty,data_uncertainty,total_uncertainty
204,1,0.050,0.950,None,None,None,0.198515
70,0,0.975,0.025,None,None,None,0.116907
131,0,0.975,0.025,None,None,None,0.116907


Uncertainty quantification is not implemented for LassoClassifierMother. predict() will be used as a fallback.
Uncertainty quantification is not implemented for LassoClassifierMother. predict() predict_proba() will be used as a fallback with entropy-based uncertainty for classification tasks.



=== Lasso_fallback_plus_entropy ===
predict() type: ndarray
predict() shape: (114,)
predict_uncertainty() shape: (114, 7)
predict_uncertainty() columns:
['pred', 'proba_0', 'proba_1', 'mean_predictions', 'knowledge_uncertainty', 'data_uncertainty', 'total_uncertainty']


,pred,proba_0,proba_1,mean_predictions,knowledge_uncertainty,data_uncertainty,total_uncertainty
204,1,0.263594,7.364059e-01,None,None,None,5.767828e-01
70,0,1.000000,1.300169e-09,None,None,None,2.790262e-08
131,0,0.999172,8.276358e-04,None,None,None,6.700973e-03



=== CatBoost_custom ===
predict() type: ndarray
predict() shape: (114,)
predict_uncertainty() shape: (114, 7)
predict_uncertainty() columns:
['pred', 'proba_0', 'proba_1', 'mean_predictions', 'knowledge_uncertainty', 'data_uncertainty', 'total_uncertainty']


,pred,proba_0,proba_1,mean_predictions,knowledge_uncertainty,data_uncertainty,total_uncertainty
204,1,0.033537,0.966463,None,0.000495,0.177159,0.177655
70,0,0.993893,0.006107,None,0.000849,0.052764,0.053613
131,0,0.983761,0.016239,None,0.001019,0.112024,0.113043


,estimator,predict_shape,uncertainty_shape
0,RandomForest_fallback_plus_entropy,"(114,)","(114, 7)"
1,Lasso_fallback_plus_entropy,"(114,)","(114, 7)"
2,CatBoost_custom,"(114,)","(114, 7)"


### Observation

For single-target classification, the standardized uncertainty frame often contains:

- `pred`
- `proba_*` columns
- uncertainty-related columns such as `knowledge_uncertainty` or `total_uncertainty`

In fallback classifiers, `mean_predictions` is typically left missing because the interface does not claim to have a meaningful ensemble mean.

## 3. Multi-target regression

Multi-target regression uses per-target column names. This is where the standardized schema becomes especially useful, because it avoids ambiguity when `predict()` returns a 2D array.

We compare one custom random forest implementation and one custom CatBoost implementation.

In [5]:
X_mt_reg, y_mt_reg = make_regression(
    n_samples=160,
    n_features=10,
    n_targets=3,
    noise=0.5,
    random_state=42,
)

X_mt_reg = pd.DataFrame(X_mt_reg, columns=[f"feature_{idx}" for idx in range(X_mt_reg.shape[1])])
y_mt_reg = pd.DataFrame(y_mt_reg, columns=["target_a", "target_b", "target_c"])

X_mt_reg_train, X_mt_reg_test, y_mt_reg_train, y_mt_reg_test = train_test_split(
    X_mt_reg, y_mt_reg, test_size=0.2, random_state=42
)

multi_target_regressors = {
    "randomforest_multitarget_custom": RandomForestRegressorMother(n_estimators=40, random_state=42),
    "catboost_multitarget_custom": CatboostRegressorMother(target_type="multi_target", iterations=40, verbose=0),
}

multi_target_regression_summaries = []
for name, model in multi_target_regressors.items():
    model.fit(X_mt_reg_train, y_mt_reg_train)
    pred_output = model.predict(X_mt_reg_test)
    uncertainty_output = model.predict_uncertainty(X_mt_reg_test)
    multi_target_regression_summaries.append(interface_summary(name, pred_output, uncertainty_output))
    show_output(name, pred_output, uncertainty_output)

pd.DataFrame(multi_target_regression_summaries)[["estimator", "predict_shape", "uncertainty_shape"]]

No loss_function specified. Using default loss function based on target type.



=== randomforest_multitarget_custom ===
predict() type: ndarray
predict() shape: (32, 3)
predict_uncertainty() shape: (32, 15)
predict_uncertainty() columns:
['target_0_pred', 'target_1_pred', 'target_2_pred', 'target_0_mean_predictions', 'target_1_mean_predictions', 'target_2_mean_predictions', 'target_0_knowledge_uncertainty', 'target_1_knowledge_uncertainty', 'target_2_knowledge_uncertainty', 'target_0_data_uncertainty', 'target_1_data_uncertainty', 'target_2_data_uncertainty', 'target_0_total_uncertainty', 'target_1_total_uncertainty', 'target_2_total_uncertainty']


,target_0_pred,target_1_pred,target_2_pred,target_0_mean_predictions,target_1_mean_predictions,target_2_mean_predictions,target_0_knowledge_uncertainty,target_1_knowledge_uncertainty,target_2_knowledge_uncertainty,target_0_data_uncertainty,target_1_data_uncertainty,target_2_data_uncertainty,target_0_total_uncertainty,target_1_total_uncertainty,target_2_total_uncertainty
105,-11.728164,-9.819990,4.316934,-11.728164,-9.819990,4.316934,None,None,None,None,None,None,237.469226,145.406908,247.836988
108,-19.954538,45.940566,97.276599,-19.954538,45.940566,97.276599,None,None,None,None,None,None,190.649216,214.356316,283.992518
141,69.468279,12.010157,-67.760694,69.468279,12.010157,-67.760694,None,None,None,None,None,None,200.462840,162.948604,133.243201



=== catboost_multitarget_custom ===
predict() type: ndarray
predict() shape: (32, 3)
predict_uncertainty() shape: (32, 15)
predict_uncertainty() columns:
['target_0_pred', 'target_1_pred', 'target_2_pred', 'target_0_mean_predictions', 'target_1_mean_predictions', 'target_2_mean_predictions', 'target_0_knowledge_uncertainty', 'target_1_knowledge_uncertainty', 'target_2_knowledge_uncertainty', 'target_0_data_uncertainty', 'target_1_data_uncertainty', 'target_2_data_uncertainty', 'target_0_total_uncertainty', 'target_1_total_uncertainty', 'target_2_total_uncertainty']


,target_0_pred,target_1_pred,target_2_pred,target_0_mean_predictions,target_1_mean_predictions,target_2_mean_predictions,target_0_knowledge_uncertainty,target_1_knowledge_uncertainty,target_2_knowledge_uncertainty,target_0_data_uncertainty,target_1_data_uncertainty,target_2_data_uncertainty,target_0_total_uncertainty,target_1_total_uncertainty,target_2_total_uncertainty
105,-0.471479,7.818296,18.765716,12.970904,12.149762,16.306019,10.628331,6.265907,1.715857,NaN,NaN,NaN,NaN,NaN,NaN
108,-7.118962,27.336959,35.115853,9.358802,24.247388,30.653868,12.821928,2.420549,5.411240,NaN,NaN,NaN,NaN,NaN,NaN
141,64.094193,27.069542,-16.201219,49.020125,23.076067,-4.874083,10.213203,3.008028,8.000762,NaN,NaN,NaN,NaN,NaN,NaN


,estimator,predict_shape,uncertainty_shape
0,randomforest_multitarget_custom,"(32, 3)","(32, 15)"
1,catboost_multitarget_custom,"(32, 3)","(32, 15)"


### Observation

In multi-target regression, the uncertainty DataFrame no longer uses a single `pred` column. Instead, it namespaces every field by target index, for example:

- `target_0_pred`
- `target_0_mean_predictions`
- `target_0_total_uncertainty`

This keeps the interface tabular and explicit.

## 4. Multi-target classification fallback

This is the easiest way to understand the generic fallback: we create a minimal classifier that implements `fit()` and `predict()`, but not a custom `predict_uncertainty()`.

Because the class inherits from `AbstractMotherPipeline`, Mother will wrap the 2D `predict()` output into the standardized multi-target uncertainty schema.

In [6]:
class MultiTargetFallbackClassifier(ml.AbstractMotherPipeline):
    _estimator_type = "classifier"

    def __init__(self):
        self.model = MultiOutputClassifier(LogisticRegression(max_iter=1000))

    def fit(self, X, y=None):
        self.model.fit(X, y)
        return self

    def predict(self, X):
        return self.model.predict(X)

    def get_hyperparameter_space(self, X, y, trial, prefix=""):
        return {}

    def set_params(self, **params):
        return self

    def get_params(self, deep=True):
        return {}


X_mt_clf, y_mt_clf = make_multilabel_classification(
    n_samples=150,
    n_features=8,
    n_classes=3,
    n_labels=2,
    random_state=42,
    return_indicator="dense",
)

X_mt_clf = pd.DataFrame(X_mt_clf, columns=[f"feature_{idx}" for idx in range(X_mt_clf.shape[1])])
y_mt_clf = pd.DataFrame(y_mt_clf, columns=[f"label_{idx}" for idx in range(y_mt_clf.shape[1])])

X_mt_clf_train, X_mt_clf_test, y_mt_clf_train, y_mt_clf_test = train_test_split(
    X_mt_clf, y_mt_clf, test_size=0.25, random_state=42
)

fallback_classifier = MultiTargetFallbackClassifier()
fallback_classifier.fit(X_mt_clf_train, y_mt_clf_train)
fallback_pred = fallback_classifier.predict(X_mt_clf_test)
fallback_uncertainty = fallback_classifier.predict_uncertainty(X_mt_clf_test)

show_output("multitarget_classification_fallback", fallback_pred, fallback_uncertainty)

pred_cols = [col for col in fallback_uncertainty.columns if col.endswith("_pred")]
mean_cols = [col for col in fallback_uncertainty.columns if col.endswith("_mean_predictions")]

print("Prediction columns are populated:", fallback_uncertainty[pred_cols].notna().all().all())
print("Mean prediction columns are missing:", fallback_uncertainty[mean_cols].isna().all().all())

Uncertainty quantification is not implemented for MultiTargetFallbackClassifier. predict() will be used as a fallback.



=== multitarget_classification_fallback ===
predict() type: ndarray
predict() shape: (38, 3)
predict_uncertainty() shape: (38, 15)
predict_uncertainty() columns:
['target_0_pred', 'target_0_mean_predictions', 'target_0_knowledge_uncertainty', 'target_0_data_uncertainty', 'target_0_total_uncertainty', 'target_1_pred', 'target_1_mean_predictions', 'target_1_knowledge_uncertainty', 'target_1_data_uncertainty', 'target_1_total_uncertainty', 'target_2_pred', 'target_2_mean_predictions', 'target_2_knowledge_uncertainty', 'target_2_data_uncertainty', 'target_2_total_uncertainty']


,target_0_pred,target_0_mean_predictions,target_0_knowledge_uncertainty,target_0_data_uncertainty,target_0_total_uncertainty,target_1_pred,target_1_mean_predictions,target_1_knowledge_uncertainty,target_1_data_uncertainty,target_1_total_uncertainty,target_2_pred,target_2_mean_predictions,target_2_knowledge_uncertainty,target_2_data_uncertainty,target_2_total_uncertainty
73,1,None,None,None,None,1,None,None,None,None,1,None,None,None,None
18,0,None,None,None,None,1,None,None,None,None,0,None,None,None,None
118,0,None,None,None,None,1,None,None,None,None,0,None,None,None,None


Prediction columns are populated: True
Mean prediction columns are missing: True


### Why this example matters

This fallback example shows an important interface detail:

- `predict()` for multi-target tasks naturally returns a 2D structure
- the fallback `predict_uncertainty()` method converts that into per-target columns
- only fields that the estimator can genuinely provide should be interpreted as meaningful

In practice, this means you should treat fallback outputs as a schema normalization mechanism, not as proof that a model has real uncertainty estimation support.

In [7]:
comparison_rows = []
comparison_rows.extend(single_target_regression_summaries)
comparison_rows.extend(single_target_classification_summaries)
comparison_rows.extend(multi_target_regression_summaries)
comparison_rows.append(interface_summary("multitarget_classification_fallback", fallback_pred, fallback_uncertainty))

comparison_df = pd.DataFrame(comparison_rows)
comparison_df[["estimator", "predict_shape", "uncertainty_shape", "uncertainty_columns"]]

,estimator,predict_shape,uncertainty_shape,uncertainty_columns
0,Lasso_fallback,"(89,)","(89, 5)","[pred, mean_predictions, knowledge_uncertainty..."
1,RandomForest_custom,"(89,)","(89, 5)","[pred, mean_predictions, knowledge_uncertainty..."
2,CatBoost_custom,"(89,)","(89, 5)","[pred, mean_predictions, knowledge_uncertainty..."
3,CatBoostGaussianProcess_custom,"(89,)","(89, 5)","[pred, mean_predictions, knowledge_uncertainty..."
4,RandomForest_fallback_plus_entropy,"(114,)","(114, 7)","[pred, proba_0, proba_1, mean_predictions, kno..."
5,Lasso_fallback_plus_entropy,"(114,)","(114, 7)","[pred, proba_0, proba_1, mean_predictions, kno..."
6,CatBoost_custom,"(114,)","(114, 7)","[pred, proba_0, proba_1, mean_predictions, kno..."
7,randomforest_multitarget_custom,"(32, 3)","(32, 15)","[target_0_pred, target_1_pred, target_2_pred, ..."
8,catboost_multitarget_custom,"(32, 3)","(32, 15)","[target_0_pred, target_1_pred, target_2_pred, ..."
9,multitarget_classification_fallback,"(38, 3)","(38, 15)","[target_0_pred, target_0_mean_predictions, tar..."


### Interpreting mother_cv output

`mother_cv()` returns a row-aligned out-of-fold table. In practice, that means each sample only receives predictions from the fold where it was held out.
This is the safest way to generate evaluation tables for uncertainty-aware workflows, because it keeps train/test separation intact while preserving Mother's standardized prediction schema.
For most data-science workflows, a useful rule of thumb is:
- use `predict()` and `predict_uncertainty()` on a final fitted model when you already have a train/test split\n
- use `mother_cv()` when you need cross-validated prediction tables for diagnostics, reporting, or model comparison

In [8]:
cv = KFold(n_splits=3, shuffle=True, random_state=42)
cv_table, cv_artifacts = mother_cv(
    RandomForestRegressorMother(n_estimators=40, random_state=42),
    X=X_reg,
    y=y_reg,
    cv=cv,
    prediction_prefix="oof_",
    return_estimators=True,
)

base_cols = [col for col in ["target", "iteration", "test_index", "cv_group"] if col in cv_table.columns]
prediction_cols = [col for col in cv_table.columns if col.startswith("oof_")]

print("Number of folds returned:", cv_table["iteration"].nunique())
print("Number of fitted estimators:", len(cv_artifacts["estimators"]))
print("Prediction columns:", prediction_cols)
display(cv_table[base_cols + prediction_cols].head())

Number of folds returned: 3
Number of fitted estimators: 3
Prediction columns: ['oof_target', 'oof_total_uncertainty']


,target,iteration,test_index,oof_target,oof_total_uncertainty
0,151,0,0,198.15,114.25
1,75,2,1,79.35,38.5
2,141,1,2,200.7,112.25
3,206,0,3,156.925,77.75
4,135,2,4,124.525,64.25


## Takeaways
- Mother aims to keep `predict_uncertainty()` structurally regular across estimators.
- The column names are standardized, but the semantics still depend on whether the estimator uses a custom uncertainty implementation or the generic fallback.
- Single-target classification often adds probability columns alongside uncertainty information.
- Multi-target tasks namespace prediction and uncertainty columns by target index.
- Generic fallback should be read as schema normalization, not as proof of a rich uncertainty model.
- `mother_cv()` is the preferred interface when you need out-of-fold predictions that preserve Mother column conventions.